In [ ]:
print("hello")

安装 Hugging Face 和 MT 相关库

In [ ]:
%pip install transformers
%pip install torch
%pip install sentencepiece
%pip install sacrebleu

测试能不能 import

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
import torch

print("ok")

NLLB 加载

In [ ]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("model loaded")

放到 GPU

In [ ]:
device = "cuda"

model = model.to(device)
print("model moved to GPU")

翻译从de到hsb

In [ ]:
print(tokenizer.convert_tokens_to_ids("hsb_Latn"))
print(tokenizer.convert_tokens_to_ids("deu_Latn"))

In [ ]:
tokenizer.src_lang = "deu_Latn"

text = "Hallo."

inputs = tokenizer(text, return_tensors="pt").to(device)

target_id = tokenizer.convert_tokens_to_ids("hsb_Latn")

with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_id,
        max_new_tokens=20,
        num_beams=4
    )

translation = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

print(translation[0])

也就是说，你这个 NLLB tokenizer 里没有 Upper Sorbian 的语言 token。

In [ ]:
print(tokenizer.convert_tokens_to_ids("hsb_Latn"))
print(tokenizer.unk_token_id)

下一步可以测试 Qwen2.5 baseline

import相关的库

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

qwen_name = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    torch_dtype="auto"
).to(device)

print("Qwen loaded")

翻译一句de到hsb

In [ ]:
prompt = """Translate the following German sentence into Upper Sorbian.
Only output the translation.

German: Guten Morgen.
Upper Sorbian:"""

inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

result = qwen_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(result)

即使 hsb 是 extremely low-resource，依靠跨语言能力生成
它输出This is a simple translation...说明：instruction following 不稳定

下一步应该做：
用真实 KDE4 数据测试

In [ ]:
de_path = r"d:\Desktop\26ss\NLP\de-hsb.txt\KDE4.de-hsb.de"
hsb_path = r"d:\Desktop\26ss\NLP\de-hsb.txt\KDE4.de-hsb.hsb"

with open(de_path, encoding="utf-8") as f:
    de_sentences = [line.strip() for line in f]

with open(hsb_path, encoding="utf-8") as f:
    hsb_references = [line.strip() for line in f]

print(len(de_sentences), len(hsb_references))

print("DE:")
print(de_sentences[0])

print("HSB:")
print(hsb_references[0])

用 Qwen 翻译真实数据里的句子
先只测试前 5 句。

In [ ]:
for i in range(5):

    german = de_sentences[i]

    prompt = f"""Translate the following German sentence into Upper Sorbian.
Only output the translation.

German: {german}

Upper Sorbian:"""

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )

    result = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 50)
    print("GERMAN:")
    print(german)

    print("\nREFERENCE:")
    print(hsb_references[i])

    print("\nQWEN:")
    print(result)

NLLB 不支持 Upper Sorbian（hsb），tokenizer 会将 hsb_Latn 识别为 unk，因此无法直接进行 de→hsb 翻译。
Qwen2.5 可以通过 prompt 生成 hsb 类文本，说明 LLM 对低资源语言具有更强的开放词表能力。
但 Qwen 输出不稳定，会重复 prompt、生成解释文本，甚至出现 hallucination 和乱码。
因此：
NLLB 更稳定，但语言覆盖有限；
Qwen 更灵活，但生成质量不稳定。

用 Qwen 对真实 de→hsb 数据做 batch translation 测试。
先用前 2000 句来评估 baseline。

In [ ]:
predictions = []

for i in range(min(2000, len(de_sentences))):

    german = de_sentences[i]

    prompt = f"""Translate the following German sentence into Upper Sorbian.
Only output the translation.

German: {german}

Upper Sorbian:"""

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    result = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # extract only generated translation
    if "Upper Sorbian:" in result:
        result = result.split("Upper Sorbian:")[-1].strip()

    predictions.append(result)

    print("=" * 40)
    print("DE:", german)
    print("REF:", hsb_references[i])
    print("PRED:", result)

现在可以开始算：

BLEU
chrF++

In [ ]:
from sacrebleu import corpus_bleu, corpus_chrf

references = [hsb_references[:len(predictions)]]

bleu = corpus_bleu(predictions, references)
chrf = corpus_chrf(predictions, references)

print("BLEU:", bleu.score)
print("chrF++:", chrf.score)


当前 Qwen baseline 翻译质量非常低。